<a href="https://colab.research.google.com/github/Ashwini9713/Digital-forensics-/blob/model-lab/exp5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!/usr/bin/env python3
"""
Forensic File Signature Auditor (single-file version)
========================================================
Compares filename extensions against actual file signatures ("magic
numbers") and structural content characteristics to catch files that
have been deliberately or accidentally mislabeled.

Run directly to see it audit a built-in set of example evidence files
covering documents, images, archives, and executables:

    python3 forensic_file_auditor.py

Or point it at real files/folders:

    python3 forensic_file_auditor.py path/to/file_or_folder [...]

Classification codes
---------------------
  MATCH        - Extension matches the detected signature/structure.
  MISMATCH     - Signature reveals a different real type than claimed.
  MASQUERADE   - Executable/script content hidden behind a
                 document/image/text extension (highest severity).
  CORRUPT      - Extension matches the type family, but structural
                 validation failed (truncated/tampered file).
  UNKNOWN_TYPE - No known signature matched.
  NO_EXTENSION - Nothing to compare against.
"""

from __future__ import annotations

import os
import sys
import struct
import zipfile
import tempfile
from dataclasses import dataclass, field
from typing import Optional


# ---------------------------------------------------------------------------
# 1. Signature database
# ---------------------------------------------------------------------------

@dataclass
class Signature:
    type_name: str
    category: str  # "document" | "image" | "archive" | "executable" | "text"
    patterns: list  # list of (offset, bytes)
    expected_extensions: set


SIGNATURES = [
    # Documents
    Signature("PDF", "document", [(0, b"%PDF-")], {".pdf"}),
    Signature("MS_OFFICE_OLE", "document", [(0, b"\xD0\xCF\x11\xE0\xA1\xB1\x1A\xE1")],
              {".doc", ".xls", ".ppt", ".msi"}),
    Signature("OOXML_ZIP", "document", [(0, b"PK\x03\x04")],
              {".docx", ".xlsx", ".pptx", ".zip", ".jar", ".apk", ".epub"}),
    Signature("RTF", "document", [(0, b"{\\rtf1")], {".rtf"}),

    # Images
    Signature("JPEG", "image", [(0, b"\xFF\xD8\xFF")], {".jpg", ".jpeg"}),
    Signature("PNG", "image", [(0, b"\x89PNG\r\n\x1a\n")], {".png"}),
    Signature("GIF87a", "image", [(0, b"GIF87a")], {".gif"}),
    Signature("GIF89a", "image", [(0, b"GIF89a")], {".gif"}),
    Signature("BMP", "image", [(0, b"BM")], {".bmp"}),
    Signature("TIFF_LE", "image", [(0, b"II*\x00")], {".tif", ".tiff"}),
    Signature("TIFF_BE", "image", [(0, b"MM\x00*")], {".tif", ".tiff"}),
    Signature("WEBP", "image", [(0, b"RIFF"), (8, b"WEBP")], {".webp"}),

    # Archives
    Signature("ZIP", "archive", [(0, b"PK\x03\x04")], {".zip", ".jar", ".apk"}),
    Signature("ZIP_EMPTY", "archive", [(0, b"PK\x05\x06")], {".zip"}),
    Signature("RAR4", "archive", [(0, b"Rar!\x1a\x07\x00")], {".rar"}),
    Signature("RAR5", "archive", [(0, b"Rar!\x1a\x07\x01\x00")], {".rar"}),
    Signature("7ZIP", "archive", [(0, b"7z\xBC\xAF\x27\x1C")], {".7z"}),
    Signature("GZIP", "archive", [(0, b"\x1F\x8B")], {".gz", ".tgz"}),
    Signature("TAR", "archive", [(257, b"ustar")], {".tar"}),

    # Executables / scripts
    Signature("PE_EXECUTABLE", "executable", [(0, b"MZ")],
              {".exe", ".dll", ".sys", ".scr", ".cpl"}),
    Signature("ELF_EXECUTABLE", "executable", [(0, b"\x7fELF")], {"", ".elf", ".bin", ".so"}),
    Signature("MACHO_32", "executable", [(0, b"\xFE\xED\xFA\xCE")], {""}),
    Signature("MACHO_64", "executable", [(0, b"\xFE\xED\xFA\xCF")], {""}),
    Signature("SHELL_SCRIPT", "executable", [(0, b"#!/")], {".sh", ".bash", ".py", ".pl", ".rb"}),
    Signature("MS_DOS_BATCH_MARKER", "executable", [(0, b"@echo")], {".bat", ".cmd"}),
]

NON_EXECUTABLE_LOOKING_EXTENSIONS = {
    ".txt", ".pdf", ".doc", ".docx", ".jpg", ".jpeg", ".png", ".gif",
    ".csv", ".log", ".xml", ".json", ".pptx", ".xlsx", ".rtf", ".html",
}


@dataclass
class AuditResult:
    path: str
    claimed_extension: str
    detected_type: Optional[str]
    detected_category: Optional[str]
    classification: str
    severity: str
    evidence: list = field(default_factory=list)
    message: str = ""

    def report(self) -> str:
        lines = [
            f"FILE:            {self.path}",
            f"Claimed ext:     {self.claimed_extension or '(none)'}",
            f"Detected type:   {self.detected_type or 'unrecognized'}"
            + (f"  [{self.detected_category}]" if self.detected_category else ""),
            f"Classification:  {self.classification}  (severity: {self.severity})",
            f"Verdict:         {self.message}",
            "Evidence:",
        ]
        for e in self.evidence:
            lines.append(f"    - {e}")
        return "\n".join(lines)


# ---------------------------------------------------------------------------
# 2. Detection engine
# ---------------------------------------------------------------------------

def _read_head(path: str, n: int = 512) -> bytes:
    with open(path, "rb") as f:
        return f.read(n)


def _match_signature(head: bytes, path: str) -> Optional[Signature]:
    for sig in SIGNATURES:
        ok = True
        for offset, pattern in sig.patterns:
            chunk = head[offset:offset + len(pattern)]
            if len(chunk) < len(pattern) or chunk != pattern:
                if sig.type_name == "TAR":
                    try:
                        with open(path, "rb") as f:
                            f.seek(257)
                            chunk2 = f.read(len(pattern))
                        if chunk2 == pattern:
                            continue
                    except OSError:
                        pass
                ok = False
                break
        if ok:
            return sig
    return None


def _looks_like_text(head: bytes) -> bool:
    if b"\x00" in head:
        return False
    try:
        head.decode("utf-8")
        return True
    except UnicodeDecodeError:
        return False


def _validate_structure(path: str, sig: Signature) -> tuple[bool, list]:
    evidence = []
    ok = True

    if sig.type_name in ("ZIP", "OOXML_ZIP"):
        try:
            with zipfile.ZipFile(path) as zf:
                bad = zf.testzip()
                if bad is not None:
                    ok = False
                    evidence.append(f"ZIP central directory reports corrupt member: {bad}")
                else:
                    names = zf.namelist()
                    evidence.append(f"ZIP central directory readable, {len(names)} entries")
                    if any(n.startswith("word/") for n in names):
                        evidence.append("Contains 'word/' -> real type is DOCX, not generic ZIP")
                    elif any(n.startswith("xl/") for n in names):
                        evidence.append("Contains 'xl/' -> real type is XLSX, not generic ZIP")
                    elif any(n.startswith("ppt/") for n in names):
                        evidence.append("Contains 'ppt/' -> real type is PPTX, not generic ZIP")
        except zipfile.BadZipFile as e:
            ok = False
            evidence.append(f"ZIP structure invalid: {e}")

    elif sig.type_name == "JPEG":
        with open(path, "rb") as f:
            data = f.read()
        if data[-2:] != b"\xFF\xD9":
            ok = False
            evidence.append("Missing standard JPEG End-Of-Image (FFD9) marker at EOF")
        else:
            evidence.append("Valid JPEG EOI marker present at EOF")

    elif sig.type_name == "PNG":
        with open(path, "rb") as f:
            data = f.read()
        if b"IEND" not in data[-64:] and b"IEND" not in data:
            ok = False
            evidence.append("No IEND chunk found - PNG stream incomplete")
        else:
            evidence.append("IEND chunk present - PNG stream terminated correctly")

    elif sig.type_name == "PE_EXECUTABLE":
        with open(path, "rb") as f:
            data = f.read(1024)
        if len(data) >= 0x40:
            pe_offset = struct.unpack("<I", data[0x3C:0x40])[0]
            evidence.append(f"DOS header points to PE header at offset 0x{pe_offset:X}")
        evidence.append("MZ DOS stub present - Windows PE/COFF executable structure")

    return ok, evidence


def _severity_for(classification: str) -> str:
    return {
        "MASQUERADE": "CRITICAL",
        "MISMATCH": "HIGH",
        "CORRUPT": "MEDIUM",
        "UNKNOWN_TYPE": "LOW",
        "NO_EXTENSION": "LOW",
    }.get(classification, "INFO")


def audit_file(path: str) -> AuditResult:
    ext = os.path.splitext(path)[1].lower()
    head = _read_head(path)
    evidence = [f"Read first {len(head)} bytes for signature analysis"]

    sig = _match_signature(head, path)

    if sig is None:
        if _looks_like_text(head):
            evidence.append("No binary magic number matched; content decodes cleanly as UTF-8/ASCII text")
            detected_type, detected_category = "PLAIN_TEXT", "text"
        else:
            evidence.append(f"No known signature matched. First bytes (hex): {head[:16].hex()}")
            detected_type, detected_category = None, None

        if detected_type == "PLAIN_TEXT" and ext in (".txt", ".csv", ".log", ".md", ".json",
                                                       ".xml", ".html", ".py", ".sh", ".pl", ".rb", ""):
            classification, message = "MATCH", "Content is plain text, consistent with claimed extension."
        elif ext == "":
            classification, message = "NO_EXTENSION", "File has no extension; cannot compare claim vs content."
        else:
            classification, message = "UNKNOWN_TYPE", "Signature not in database; type could not be confirmed or refuted."

        return AuditResult(path, ext, detected_type, detected_category,
                            classification, _severity_for(classification), evidence, message)

    evidence.append(
        f"Matched signature '{sig.type_name}' "
        f"(pattern {sig.patterns[0][1][:8]!r} at offset {sig.patterns[0][0]})"
    )
    struct_ok, struct_evidence = _validate_structure(path, sig)
    evidence.extend(struct_evidence)

    extension_expected = ext in sig.expected_extensions or (ext == "" and "" in sig.expected_extensions)

    if extension_expected and not struct_ok:
        classification = "CORRUPT"
        message = (f"Extension matches expected family for {sig.type_name}, but structural "
                    f"validation failed — file may be truncated, tampered with, or deliberately damaged.")
    elif extension_expected and struct_ok:
        classification = "MATCH"
        message = f"Signature and structure confirm genuine {sig.type_name}; extension is accurate."
    else:
        if sig.category == "executable" and ext in NON_EXECUTABLE_LOOKING_EXTENSIONS:
            classification = "MASQUERADE"
            message = (f"File is a real {sig.type_name} disguised with a '{ext or '(none)'}' extension. "
                        f"Classic technique for smuggling executable content past filters or "
                        f"casual inspection — treat as high-priority evidence.")
        else:
            classification = "MISMATCH"
            message = (f"Content signature identifies this as {sig.type_name} "
                        f"({sig.category}), not what the '{ext or '(none)'}' extension claims.")

    return AuditResult(path, ext, sig.type_name, sig.category,
                        classification, _severity_for(classification), evidence, message)


def audit_path(target: str) -> list:
    results = []
    if os.path.isdir(target):
        for root, _, files in os.walk(target):
            for fn in files:
                results.append(audit_file(os.path.join(root, fn)))
    else:
        results.append(audit_file(target))
    return results


def print_report(results: list) -> None:
    flagged = [r for r in results if r.classification != "MATCH"]
    print("=" * 78)
    print(f"FILE SIGNATURE AUDIT — {len(results)} file(s) examined, {len(flagged)} flagged")
    print("=" * 78)
    for r in results:
        print()
        print(r.report())
        print("-" * 78)
    print("\nSUMMARY")
    for r in results:
        marker = "OK " if r.classification == "MATCH" else "!! "
        print(f"  [{marker}{r.classification:<12}] {r.path}")


# ---------------------------------------------------------------------------
# 3. Built-in example evidence (documents, images, archives, executables)
# ---------------------------------------------------------------------------

def _build_demo_files(tmp_dir: str) -> list:
    paths = []

    def w(name, data):
        p = os.path.join(tmp_dir, name)
        with open(p, "wb") as f:
            f.write(data)
        paths.append(p)

    # Genuine PDF, correctly named
    w("contract_final.pdf", b"%PDF-1.4\n%...\n1 0 obj<< >>endobj\n%%EOF")

    # Genuine PNG renamed to .docx -> MISMATCH
    w("quarterly_report.docx",
      b"\x89PNG\r\n\x1a\n" + b"\x00\x00\x00\rIHDR" + b"\x00" * 13 + b"junkdata" + b"IEND\xaeB`\x82")

    # Genuine JPEG, correctly named
    w("scene_photo.jpg", b"\xFF\xD8\xFF\xE0" + b"\x00" * 40 + b"\xFF\xD9")

    # JPEG missing EOI marker -> CORRUPT
    w("tampered_photo.jpg", b"\xFF\xD8\xFF\xE0" + b"\x00" * 40)

    # Genuine ZIP, correctly named
    zpath = os.path.join(tmp_dir, "case_files.zip")
    with zipfile.ZipFile(zpath, "w") as zf:
        zf.writestr("notes.txt", "evidence log")
    paths.append(zpath)

    # Windows PE executable disguised as "invoice.pdf" -> MASQUERADE
    pe_stub = bytearray(b"MZ" + b"\x00" * 58 + struct.pack("<I", 0x80))
    pe_stub += b"\x00" * (0x80 - len(pe_stub))
    pe_stub += b"PE\x00\x00" + b"\x00" * 20
    w("invoice.pdf", bytes(pe_stub))

    # Shell script renamed to .txt -> MASQUERADE
    w("readme.txt", b"#!/bin/bash\nrm -rf /tmp/evidence_cache\necho done\n")

    # Genuine DOCX (OOXML zip with word/ dir), correctly named
    dpath = os.path.join(tmp_dir, "memo.docx")
    with zipfile.ZipFile(dpath, "w") as zf:
        zf.writestr("word/document.xml", "<xml>hello</xml>")
        zf.writestr("[Content_Types].xml", "<Types/>")
    paths.append(dpath)

    # Plain text, correctly named
    w("chain_of_custody.txt", "Item received 2026-08-25, logged by investigator.\n".encode())

    # RAR archive renamed to .jpg -> MISMATCH
    w("vacation.jpg", b"Rar!\x1a\x07\x00" + b"\x00" * 30)

    return paths


# ---------------------------------------------------------------------------
# 4. CLI
# ---------------------------------------------------------------------------

def main(argv):
    if argv:
        results = []
        for target in argv:
            if not os.path.exists(target):
                print(f"[!] Path not found, skipping: {target}")
                continue
            results.extend(audit_path(target))
        print_report(results)
        return 0

    # No arguments -> run the built-in demo covering all four categories
    print("No path given — running built-in demo evidence set "
          "(documents, images, archives, executables).\n")
    with tempfile.TemporaryDirectory() as tmp_dir:
        demo_paths = _build_demo_files(tmp_dir)
        results = [audit_file(p) for p in demo_paths]
        print_report(results)
    return 0


if __name__ == "__main__":
    sys.exit(main(sys.argv[1:]))

[!] Path not found, skipping: -f
FILE SIGNATURE AUDIT — 1 file(s) examined, 0 flagged

FILE:            /root/.local/share/jupyter/runtime/kernel-0742a975-9e3b-4da4-b507-4d0e1d31d2b2.json
Claimed ext:     .json
Detected type:   PLAIN_TEXT  [text]
Classification:  MATCH  (severity: INFO)
Verdict:         Content is plain text, consistent with claimed extension.
Evidence:
    - Read first 283 bytes for signature analysis
    - No binary magic number matched; content decodes cleanly as UTF-8/ASCII text
------------------------------------------------------------------------------

SUMMARY
  [OK MATCH       ] /root/.local/share/jupyter/runtime/kernel-0742a975-9e3b-4da4-b507-4d0e1d31d2b2.json


SystemExit: 0

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
